# Homework 3

## Time Series Classification Part 1: Feature Creation/Extraction

In [1]:
import os
import pandas as pd
import numpy as np

### 1(a) Download and Load the AReM Dataset

In [2]:
folders = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

for folder in folders:
    print(folder, os.listdir(folder))

bending1 ['dataset1.csv', 'dataset2.csv', 'dataset3.csv', 'dataset4.csv', 'dataset5.csv', 'dataset6.csv', 'dataset7.csv']
bending2 ['dataset1.csv', 'dataset2.csv', 'dataset3.csv', 'dataset4.csv', 'dataset5.csv', 'dataset6.csv']
cycling ['dataset1.csv', 'dataset10.csv', 'dataset11.csv', 'dataset12.csv', 'dataset13.csv', 'dataset14.csv', 'dataset15.csv', 'dataset2.csv', 'dataset3.csv', 'dataset4.csv', 'dataset5.csv', 'dataset6.csv', 'dataset7.csv', 'dataset8.csv', 'dataset9.csv']
lying ['dataset1.csv', 'dataset10.csv', 'dataset11.csv', 'dataset12.csv', 'dataset13.csv', 'dataset14.csv', 'dataset15.csv', 'dataset2.csv', 'dataset3.csv', 'dataset4.csv', 'dataset5.csv', 'dataset6.csv', 'dataset7.csv', 'dataset8.csv', 'dataset9.csv']
sitting ['dataset1.csv', 'dataset10.csv', 'dataset11.csv', 'dataset12.csv', 'dataset13.csv', 'dataset14.csv', 'dataset15.csv', 'dataset2.csv', 'dataset3.csv', 'dataset4.csv', 'dataset5.csv', 'dataset6.csv', 'dataset7.csv', 'dataset8.csv', 'dataset9.csv']
standing 

In [3]:
column_names = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

### 1(b) Train/Test Split

In [4]:
test_files = {}

for folder in folders:
    if folder in ["bending1", "bending2"]:
        test_files[folder] = ["dataset1.csv", "dataset2.csv"]
    else:
        test_files[folder] = ["dataset1.csv", "dataset2.csv", "dataset3.csv"]

test_files

{'bending1': ['dataset1.csv', 'dataset2.csv'],
 'bending2': ['dataset1.csv', 'dataset2.csv'],
 'cycling': ['dataset1.csv', 'dataset2.csv', 'dataset3.csv'],
 'lying': ['dataset1.csv', 'dataset2.csv', 'dataset3.csv'],
 'sitting': ['dataset1.csv', 'dataset2.csv', 'dataset3.csv'],
 'standing': ['dataset1.csv', 'dataset2.csv', 'dataset3.csv'],
 'walking': ['dataset1.csv', 'dataset2.csv', 'dataset3.csv']}

In [5]:
train_data = []
test_data = []

for folder in folders:
    for file in os.listdir(folder):
        if file.endswith(".csv"):
            file_info = (folder, file)

            if file in test_files[folder]:
                test_data.append(file_info)
            else:
                train_data.append(file_info)

print("Number of training instances:", len(train_data))
print("Number of test instances:", len(test_data))
print("Total instances:", len(train_data) + len(test_data))

Number of training instances: 69
Number of test instances: 19
Total instances: 88


### 1(c) Feature Extraction

#### 1(c)i Time-Domain Features
Common time-domain features used in time series classification include minimum, maximum, mean, median, standard deviation, variance, first quartile, third quartile, range, and interquartile range.

#### 1(c)ii Extract Time-Domain Features

In [6]:
time_series_columns = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

def extract_features(data):
    features = {}

    for i, column in enumerate(time_series_columns, start=1):
        series = data[column]

        features[f"min{i}"] = series.min()
        features[f"max{i}"] = series.max()
        features[f"mean{i}"] = series.mean()
        features[f"median{i}"] = series.median()
        features[f"std{i}"] = series.std()
        features[f"1st_quart{i}"] = series.quantile(0.25)
        features[f"3rd_quart{i}"] = series.quantile(0.75)

    return features

In [7]:
all_features = []

for folder in folders:
    for file in os.listdir(folder):
        if file.endswith(".csv"):
            file_path = os.path.join(folder, file)

            with open(file_path, "r") as f:
                lines = f.readlines()

            cleaned_lines = [line.rstrip().rstrip(",") for line in lines[5:]]

            from io import StringIO
            data = pd.read_csv(
                StringIO("\n".join(cleaned_lines)),
                header=None,
                names=column_names
            )

            features = extract_features(data)
            features["activity"] = folder
            features["dataset"] = file

            all_features.append(features)

feature_df = pd.DataFrame(all_features)

print("Number of instances:", feature_df.shape[0])
print("Number of extracted features:", 42)

feature_df.head()

Number of instances: 88
Number of extracted features: 42


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6,activity,dataset
0,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,...,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30,bending1,dataset1.csv
1,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,...,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30,bending1,dataset2.csv
2,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94,bending1,dataset3.csv
3,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00,bending1,dataset4.csv
4,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50,bending1,dataset5.csv


#### 1(c)iii Standard Deviations and 90% Bootstrap Confidence Intervals

In [8]:
feature_columns = [col for col in feature_df.columns 
                   if col not in ["activity", "dataset"]]

results = []

np.random.seed(42)

for feature in feature_columns:
    values = feature_df[feature].dropna().values

    std_estimate = np.std(values, ddof=1)

    bootstrap_stds = []
    for _ in range(1000):
        bootstrap_sample = np.random.choice(
            values,
            size=len(values),
            replace=True
        )
        bootstrap_stds.append(
            np.std(bootstrap_sample, ddof=1)
        )

    lower_bound = np.percentile(bootstrap_stds, 5)
    upper_bound = np.percentile(bootstrap_stds, 95)

    results.append({
        "Feature": feature,
        "Standard Deviation": std_estimate,
        "90% CI Lower": lower_bound,
        "90% CI Upper": upper_bound
    })

std_ci_df = pd.DataFrame(results)

std_ci_df

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.624011,8.326581,10.737514
1,max1,4.207745,3.143285,5.123189
2,mean1,5.276431,4.667244,5.847694
3,median1,5.386624,4.710145,5.969270
4,std1,1.771282,1.568423,1.946107
5,1st_quart1,6.127846,5.516241,6.638798
6,3rd_quart1,5.031028,4.166218,5.707824
7,min2,0.000000,0.000000,0.000000
8,max2,5.059656,4.614432,5.398190
9,mean2,1.577908,1.416870,1.712550


#### 1(c)iv Three Most Important Time-Domain Features
I would select minimum, mean, and maximum as the three most important time-domain features. The minimum and maximum capture the range and extreme values of the time series, while the mean represents its overall central tendency.

## 2. ISLR 3.7.4

### (a)

The cubic regression will have training RSS less than or equal to that of the linear regression. The cubic model contains the linear model as a special case when the coefficients of \(X^2\) and \(X^3\) are zero. Therefore, adding these extra predictors cannot increase the training RSS, even though the true relationship is linear.

### (b)

The linear regression is expected to have a lower test RSS. Since the true relationship between \(X\) and \(Y\) is linear, the cubic regression introduces unnecessary flexibility, which can lead to overfitting and higher variance. Therefore, the simpler linear model is expected to perform better on test data.

### (c)

The cubic regression will have training RSS less than or equal to that of the linear regression. Since the cubic model is more flexible and contains the linear model as a special case, it can fit the training data at least as well as the linear model, regardless of the true relationship.

### (d)

There is not enough information to determine which model will have a lower test RSS. Since the true relationship is nonlinear, the cubic model may perform better if it approximates the true relationship well. However, if the true relationship is only slightly nonlinear or has a form that is not well represented by a cubic function, the linear model may have lower test RSS due to its lower variance.